# DeepSeek V3.1 Evaluation on WebGIS Function Call Translation

This notebook evaluates DeepSeek V3.1 (via OpenRouter API) on translating natural language queries into geospatial function calls.

**Features:**
- Checkpoint/resume capability (saves progress every 10 samples)
- Comprehensive logging to `logs/` folder
- Network error handling with retries
- Automatic resume from last checkpoint


In [7]:
# Install required packages
#%pip install requests python-dotenv evaluate rouge-score nltk editdistance pandas 

In [8]:
import json
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv
from evaluate import load
import editdistance
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from tqdm import tqdm
import logging
from datetime import datetime
from pathlib import Path

# Load environment variables
load_dotenv()

# OpenRouter API configuration
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "deepseek/deepseek-chat-v3.1"  # DeepSeek V3.1

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found. Please set it in .env file or environment variable.")

# Setup logging
log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)
log_file = log_dir / f"deepseek_evaluation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file, encoding='utf-8'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)
logger.info("="*60)
logger.info("DeepSeek V3.1 Evaluation Started")
logger.info("="*60)
logger.info(f"Log file: {log_file}")
logger.info(f"Model: {MODEL}")


2025-11-20 17:23:35,823 - INFO - ============================================================
2025-11-20 17:23:35,826 - INFO - DeepSeek V3.1 Evaluation Started
2025-11-20 17:23:35,827 - INFO - ============================================================
2025-11-20 17:23:35,829 - INFO - Log file: logs\deepseek_evaluation_20251120_172335.log
2025-11-20 17:23:35,833 - INFO - Model: deepseek/deepseek-chat-v3.1


In [9]:
# Load test dataset
test_file = "output/train_test_split_clean/dataset_1_operation_test.json"

with open(test_file, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_samples = test_data["samples"]
logger.info(f"Loaded {len(test_samples)} test samples from {test_file}")
print(f"Loaded {len(test_samples)} test samples")


2025-11-20 17:23:37,694 - INFO - Loaded 1332 test samples from output/train_test_split_clean/dataset_1_operation_test.json


Loaded 1332 test samples


In [10]:
# Create few-shot prompt based on training data examples
# This prompt includes examples for all 39 functions from function_list

FEW_SHOT_PROMPT = """You are an expert system that translates user queries into geospatial function calls. Here are some examples:

User: Start tracking my position.
Function Call: LocateUser()

User: Add a feature to the 'archaeological_sites' layer. The point location is 'POINT(-111.89 40.76)' and the properties object should contain {{ site_id: 'AZ-123', period: 'Archaic' }}.
Function Call: addFeature('archaeological_sites', 'POINT(-111.89 40.76)', {{ site_id: 'AZ-123', period: 'Archaic' }})

User: Zoom in to focus on the details.
Function Call: ZoomIn()

User: Can you reduce the zoom level?
Function Call: ZoomOut()

User: I need to correct my last request; zoom to level 12 for the 'watersheds_d4e7a1' data, not level 9.
Function Call: ZoomTo(12)

User: Yes, please fly to the layer named 'public_parks_kj7m'. Zoom level should be 14, and the transition should take 3200 milliseconds.
Function Call: FlyTo('public_parks_kj7m', 14, 3200)

User: Rotate to 300 degrees.
Function Call: RotateMap(300)

User: To finalize, execute the command that resets the rotation of the current viewport to its original state.
Function Call: ResetRotation()

User: Let's start over. Please fit the map to display all features in the 'wetlands_f8p2' layer.
Function Call: FitToLayer('wetlands_f8p2')

User: Change to fullscreen.
Function Call: ToggleFullScreen()

User: Show a popup at the coordinate 40.7128, -74.0060 with the message 'Landmark: Statue of Liberty'.
Function Call: ShowPopup([40.7128, -74.0060], 'Landmark: Statue of Liberty')

User: Generate a PDF of what we're seeing on the map now.
Function Call: printMap()

User: Correction: the base layer should be Bing, not OSM. Its name is 'satellite_view_789'.
Function Call: addBaseLayer('satellite_view_789', 'BING')

User: I made an error. The correct server for the votingPrecincts layer is https://elections.department.gov/geowebcache/wms.
Function Call: addWMS('votingPrecincts', 'https://elections.department.gov/geowebcache/wms')

User: Just to be sure, the command is addWFS('fire_hydrants_J9R', 'https://water.department/wfs', 'fh:hydrants'), correct?
Function Call: addWFS('fire_hydrants_J9R', 'https://water.department/wfs', 'fh:hydrants')

User: Fetch the KML at https://maps.site.com/areas.kml and store it as layer zones_5b.
Function Call: importVectorLayer('zones_5b', 'https://maps.site.com/areas.kml', 'format=KML')

User: Affirmative, your understanding is correct. Execute removal of the 'census_tracts_2020' layer.
Function Call: removeLayer('census_tracts_2020')

User: Pull up a list of every active layer in this project.
Function Call: listLayers()

User: I need to fix the display. Set the z-index of the 'subway_stations_entrances' layer to 65.
Function Call: setLayerZIndex('subway_stations_entrances', 65)

User: Hold on, I gave you the wrong format. For layer 'wetlands_inventory_f1e9', use format=KML, not GeoJSON.
Function Call: exportVectorLayer('wetlands_inventory_f1e9', 'format=KML')

User: Please remove all the features from layer 'incident_reports'.
Function Call: clearLayer('incident_reports')

User: Display the features contained within the layer called 'urban_parks_5A'.
Function Call: showLayerFeatures('urban_parks_5A')

User: Select the feature in the 'city_limit_a5z' layer where the name attribute is equal to 'Springfield'.
Function Call: selectFeature('city_limit_a5z', "name='Springfield'")

User: Create a 500-meter buffer around the polygon POLYGON ((10 10, 20 20, 30 10, 10 10)).
Function Call: BufferGeometry('POLYGON ((10 10, 20 20, 30 10, 10 10))', 500)

User: Generate the centroid for the watershed polygon defined by the following WKT: 'POLYGON ((50 50, 60 50, 60 60, 50 60, 50 50))'.
Function Call: Centroid('POLYGON ((50 50, 60 50, 60 60, 50 60, 50 50))')

User: No, not that geometry. I need the extent for 'GEOMETRYCOLLECTION(POINT(5 10), POLYGON((0 0, 8 0, 8 8, 0 8, 0 0)))'.
Function Call: GetExtent('GEOMETRYCOLLECTION(POINT(5 10), POLYGON((0 0, 8 0, 8 8, 0 8, 0 0)))')

User: I need confirmation on the length of this specific railway: LINESTRING (-75.5 40.2, -75.3 40.4, -75.1 40.6).
Function Call: MeasureLength('LINESTRING (-75.5 40.2, -75.3 40.4, -75.1 40.6)')

User: Measure the area of the polygon defined by this Well-Known Text: POLYGON ((151.2093 -33.8688, 151.2103 -33.8688, 151.2103 -33.8678, 151.2093 -33.8678, 151.2093 -33.8688)).
Function Call: MeasureArea('POLYGON ((151.2093 -33.8688, 151.2103 -33.8688, 151.2103 -33.8678, 151.2093 -33.8678, 151.2093 -33.8688))')

User: Generate a buffer zone around the features in the 'parcels_83b' layer.
Function Call: BufferLayer('parcels_83b')

User: Make the fill color of feature_set_7 royalblue.
Function Call: SetFillColor('feature_set_7', 'royalblue')

User: Actually, scratch the previous color. For 'power_grid_primary', set the stroke to 'rgb(150, 25, 150)'.
Function Call: SetStrokeColor('power_grid_primary', 'rgb(150, 25, 150)')

User: Apply a stroke width of 1.5 to the outlines of the polygons in the 'land_use_zoning' layer.
Function Call: SetStrokeWidth('land_use_zoning', 1.5)

User: For visual analysis, set the opacity of the 'flood_risk_zones' layer to 50%.
Function Call: SetOpacity('flood_risk_zones', 0.5)

User: To be precise, the hotspot size for the 'sensor-network-ef9' layer needs to be 18.
Function Call: SetPointSize('sensor-network-ef9', 18)

User: Actually, I need you to use the new icon URL 'https://cdn.io/geo/marker-blue.png' for layer 'selected_points', scaled down to 0.75.
Function Call: SetIconImage('selected_points', 'https://cdn.io/geo/marker-blue.png', 0.75)

User: Deactivate the icon image from 'https://img.server/icn/default.png' for the 'default_markers' layer at a scale of 1.
Function Call: RemoveIconImage('default_markers', 'https://img.server/icn/default.png', 1)

User: For the layer containing the geometry 'POINT(-121.89 37.34)', which is named 'sample_point_1', use the 'description' attribute for its label.
Function Call: SetLabel('sample_point_1', 'description')

User: To clarify my last request, I meant set the label color for the landuse_zoning_45 layer to 'darkred'.
Function Call: SetLabelColor('landuse_zoning_45', 'darkred')

User: Strip the label from layer aB9cN3.
Function Call: removeLabel('aB9cN3')

User: {user_query}
Function Call:"""

logger.info("Few-shot prompt created with 39 examples covering all functions")
print("Few-shot prompt created with 39 examples covering all functions")


2025-11-20 17:23:39,248 - INFO - Few-shot prompt created with 39 examples covering all functions


Few-shot prompt created with 39 examples covering all functions


In [11]:
# OPTIONAL: Clear checkpoint to start fresh
# Set CLEAR_CHECKPOINT = True to delete the checkpoint and start from the beginning

CLEAR_CHECKPOINT = True  # Set to True to clear checkpoint, False to keep it

if CLEAR_CHECKPOINT:
    CHECKPOINT_FILE = "deepseek_evaluation_checkpoint.json"
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        logger.info(f"Checkpoint file {CHECKPOINT_FILE} deleted. Starting fresh.")
        print(f"✓ Checkpoint cleared. Starting fresh evaluation.")
    else:
        logger.info("No checkpoint file found.")
        print("No checkpoint file found.")
else:
    # Check if checkpoint exists and if it has all errors
    CHECKPOINT_FILE = "deepseek_evaluation_checkpoint.json"
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            checkpoint = json.load(f)
        stats = checkpoint.get("stats", {})
        errors = stats.get("errors", 0)
        success = stats.get("success", 0)
        total = len(checkpoint.get("predictions", []))
        
        if total > 0 and success == 0 and errors == total:
            logger.warning(f"Checkpoint found with {errors} errors and 0 successes. Consider clearing it.")
            print(f"\n⚠ WARNING: Checkpoint has {errors} errors and 0 successes.")
            print(f"  If you want to start fresh, set CLEAR_CHECKPOINT = True in the cell above and re-run.")
        else:
            print(f"\n✓ Checkpoint found: {success} success, {errors} errors out of {total} samples")
    else:
        print("\n✓ No checkpoint found. Will start fresh evaluation.")


2025-11-20 17:23:40,464 - INFO - Checkpoint file deepseek_evaluation_checkpoint.json deleted. Starting fresh.


✓ Checkpoint cleared. Starting fresh evaluation.


In [12]:
# Checkpoint file paths
CHECKPOINT_FILE = "deepseek_evaluation_checkpoint.json"
FINAL_PREDICTIONS_FILE = "deepseek_predictions.json"
FINAL_RESULTS_FILE = "deepseek_evaluation_results.json"

def load_checkpoint():
    """Load checkpoint if it exists"""
    if os.path.exists(CHECKPOINT_FILE):
        logger.info(f"Found checkpoint file: {CHECKPOINT_FILE}")
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            checkpoint = json.load(f)
        logger.info(f"Loaded checkpoint: {checkpoint['last_processed_index'] + 1} samples already processed")
        return checkpoint
    return None

def save_checkpoint(predictions, references, inputs, last_index, stats):
    """Save checkpoint with current progress"""
    checkpoint = {
        "predictions": predictions,
        "references": references,
        "inputs": inputs,
        "last_processed_index": last_index,
        "stats": stats,
        "timestamp": datetime.now().isoformat()
    }
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, indent=2, ensure_ascii=False)
    logger.debug(f"Checkpoint saved at index {last_index}")

def call_openrouter_api(user_query: str, max_retries: int = 5, base_delay: int = 2) -> str:
    """
    Call OpenRouter API with DeepSeek V3.1 to translate user query to function call.
    Includes retry logic with exponential backoff for network issues.
    """
    prompt = FEW_SHOT_PROMPT.format(user_query=user_query)
    
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://github.com",
        "X-Title": "WebGIS Function Call Translation"
    }
    
    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0,  # Minimize randomness for consistent outputs
        "max_tokens": 64,   # Limit output length
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.post(OPENROUTER_API_URL, headers=headers, json=payload, timeout=120)
            response.raise_for_status()
            
            result = response.json()
            
            if 'choices' in result and len(result['choices']) > 0:
                return result['choices'][0]['message']['content'].strip()
            else:
                raise ValueError("Unexpected API response format")
                
        except requests.exceptions.Timeout as e:
            logger.warning(f"Request timeout (attempt {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                delay = base_delay * (2 ** attempt)
                logger.info(f"Waiting {delay} seconds before retry...")
                time.sleep(delay)
            else:
                raise Exception(f"API call failed after {max_retries} attempts due to timeout: {e}")
                
        except requests.exceptions.ConnectionError as e:
            logger.warning(f"Connection error (attempt {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                delay = base_delay * (2 ** attempt)
                logger.info(f"Waiting {delay} seconds before retry...")
                time.sleep(delay)
            else:
                raise Exception(f"API call failed after {max_retries} attempts due to connection error: {e}")
                
        except requests.exceptions.RequestException as e:
            logger.warning(f"Request error (attempt {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                delay = base_delay * (2 ** attempt)
                logger.info(f"Waiting {delay} seconds before retry...")
                time.sleep(delay)
            else:
                raise Exception(f"API call failed after {max_retries} attempts: {e}")
    
    return ""


In [13]:
# Initialize or load from checkpoint
checkpoint = load_checkpoint()

if checkpoint:
    predictions = checkpoint["predictions"]
    references = checkpoint["references"]
    inputs = checkpoint["inputs"]
    start_index = checkpoint["last_processed_index"] + 1
    stats = checkpoint.get("stats", {"success": 0, "errors": 0, "total_time": 0})
    logger.info(f"Resuming from checkpoint: starting at index {start_index}")
    logger.info(f"Already processed: {len(predictions)} samples")
    print(f"\n✓ Resuming from checkpoint: {len(predictions)} samples already processed")
    print(f"  Starting from index {start_index}/{len(test_samples)}")
else:
    predictions = []
    references = []
    inputs = []
    start_index = 0
    stats = {"success": 0, "errors": 0, "total_time": 0}
    logger.info("Starting fresh evaluation (no checkpoint found)")
    print("\nStarting fresh evaluation")

# Make predictions for remaining test samples
remaining_samples = len(test_samples) - start_index
logger.info(f"Processing {remaining_samples} remaining samples")
print(f"\nMaking predictions for {remaining_samples} remaining test samples...")
print("This may take a while due to API rate limits...")
print(f"Checkpoint will be saved every 10 samples to: {CHECKPOINT_FILE}")

CHECKPOINT_INTERVAL = 10  # Save checkpoint every 10 samples
RATE_LIMIT_DELAY = 1  # Wait 1 second between requests

start_time = time.time()

for i in range(start_index, len(test_samples)):
    sample = test_samples[i]
    user_query = sample["input"]
    expected_output = sample["output"]
    
    sample_start_time = time.time()
    
    try:
        predicted_output = call_openrouter_api(user_query)
        predictions.append(predicted_output)
        references.append(expected_output)
        inputs.append(user_query)
        stats["success"] += 1
        
        sample_time = time.time() - sample_start_time
        stats["total_time"] += sample_time
        
        logger.info(f"Sample {i+1}/{len(test_samples)}: Success (took {sample_time:.2f}s)")
        
        # Rate limiting: wait between requests
        time.sleep(RATE_LIMIT_DELAY)
        
    except Exception as e:
        error_msg = str(e)
        logger.error(f"Sample {i+1}/{len(test_samples)}: Error - {error_msg}")
        predictions.append("")
        references.append(expected_output)
        inputs.append(user_query)
        stats["errors"] += 1
    
    # Save checkpoint every CHECKPOINT_INTERVAL samples
    if (i + 1) % CHECKPOINT_INTERVAL == 0:
        save_checkpoint(predictions, references, inputs, i, stats)
        elapsed = time.time() - start_time
        avg_time = stats["total_time"] / max(stats["success"], 1)
        remaining = (len(test_samples) - i - 1) * avg_time
        logger.info(f"Checkpoint saved at sample {i + 1}. Progress: {i+1}/{len(test_samples)} ({100*(i+1)/len(test_samples):.1f}%)")
        logger.info(f"Stats: {stats['success']} success, {stats['errors']} errors")
        logger.info(f"Estimated time remaining: {remaining/60:.1f} minutes")
        print(f"\n✓ Checkpoint saved at sample {i + 1}/{len(test_samples)} ({100*(i+1)/len(test_samples):.1f}%)")

# Final checkpoint save
save_checkpoint(predictions, references, inputs, len(test_samples) - 1, stats)

total_time = time.time() - start_time
logger.info(f"Completed predictions for {len(predictions)} samples")
logger.info(f"Total time: {total_time/60:.2f} minutes")
logger.info(f"Success: {stats['success']}, Errors: {stats['errors']}")
print(f"\n✓ Completed predictions for {len(predictions)} samples")
print(f"  Total time: {total_time/60:.2f} minutes")
print(f"  Success: {stats['success']}, Errors: {stats['errors']}")


2025-11-20 17:23:54,400 - INFO - Starting fresh evaluation (no checkpoint found)
2025-11-20 17:23:54,402 - INFO - Processing 1332 remaining samples



Starting fresh evaluation

Making predictions for 1332 remaining test samples...
This may take a while due to API rate limits...
Checkpoint will be saved every 10 samples to: deepseek_evaluation_checkpoint.json


2025-11-20 17:23:56,890 - INFO - Sample 1/1332: Success (took 2.48s)
2025-11-20 17:24:00,735 - INFO - Sample 2/1332: Success (took 2.83s)
2025-11-20 17:24:04,417 - INFO - Sample 3/1332: Success (took 2.67s)
2025-11-20 17:24:09,571 - INFO - Sample 4/1332: Success (took 4.14s)
2025-11-20 17:24:12,903 - INFO - Sample 5/1332: Success (took 2.31s)
2025-11-20 17:24:17,096 - INFO - Sample 6/1332: Success (took 3.17s)
2025-11-20 17:24:22,758 - INFO - Sample 7/1332: Success (took 4.65s)
2025-11-20 17:24:29,576 - INFO - Sample 8/1332: Success (took 5.81s)
2025-11-20 17:24:33,529 - INFO - Sample 9/1332: Success (took 2.94s)
2025-11-20 17:24:36,484 - INFO - Sample 10/1332: Success (took 1.94s)
2025-11-20 17:24:37,495 - INFO - Checkpoint saved at sample 10. Progress: 10/1332 (0.8%)
2025-11-20 17:24:37,497 - INFO - Stats: 10 success, 0 errors
2025-11-20 17:24:37,498 - INFO - Estimated time remaining: 72.6 minutes



✓ Checkpoint saved at sample 10/1332 (0.8%)


2025-11-20 17:24:41,269 - INFO - Sample 11/1332: Success (took 3.77s)
2025-11-20 17:24:52,428 - INFO - Sample 12/1332: Success (took 10.15s)
2025-11-20 17:24:56,623 - INFO - Sample 13/1332: Success (took 3.18s)
2025-11-20 17:25:01,329 - INFO - Sample 14/1332: Success (took 3.67s)
2025-11-20 17:25:05,127 - INFO - Sample 15/1332: Success (took 2.79s)
2025-11-20 17:25:09,695 - INFO - Sample 16/1332: Success (took 3.55s)
2025-11-20 17:25:12,599 - INFO - Sample 17/1332: Success (took 1.89s)
2025-11-20 17:25:16,903 - INFO - Sample 18/1332: Success (took 3.30s)
2025-11-20 17:25:21,065 - INFO - Sample 19/1332: Success (took 3.15s)
2025-11-20 17:25:26,794 - INFO - Sample 20/1332: Success (took 4.71s)
2025-11-20 17:25:27,811 - INFO - Checkpoint saved at sample 20. Progress: 20/1332 (1.5%)
2025-11-20 17:25:27,813 - INFO - Stats: 20 success, 0 errors
2025-11-20 17:25:27,815 - INFO - Estimated time remaining: 79.9 minutes


...


✓ Checkpoint saved at sample 1320/1332 (99.1%)


2025-11-20 19:35:05,458 - INFO - Sample 1321/1332: Success (took 2.17s)
2025-11-20 19:35:11,178 - INFO - Sample 1322/1332: Success (took 4.71s)
2025-11-20 19:35:15,346 - INFO - Sample 1323/1332: Success (took 3.15s)
2025-11-20 19:35:25,219 - INFO - Sample 1324/1332: Success (took 8.87s)
2025-11-20 19:35:28,306 - INFO - Sample 1325/1332: Success (took 2.07s)
2025-11-20 19:35:30,994 - INFO - Sample 1326/1332: Success (took 1.68s)
2025-11-20 19:35:38,443 - INFO - Sample 1327/1332: Success (took 6.43s)
2025-11-20 19:35:41,503 - INFO - Sample 1328/1332: Success (took 2.05s)
2025-11-20 19:35:44,610 - INFO - Sample 1329/1332: Success (took 2.10s)
2025-11-20 19:35:48,078 - INFO - Sample 1330/1332: Success (took 2.45s)
2025-11-20 19:35:49,098 - INFO - Checkpoint saved at sample 1330. Progress: 1330/1332 (99.8%)
2025-11-20 19:35:49,099 - INFO - Stats: 1330 success, 0 errors
2025-11-20 19:35:49,101 - INFO - Estimated time remaining: 0.2 minutes



✓ Checkpoint saved at sample 1330/1332 (99.8%)


2025-11-20 19:35:50,742 - INFO - Sample 1331/1332: Success (took 1.64s)
2025-11-20 19:35:54,111 - INFO - Sample 1332/1332: Success (took 2.36s)
2025-11-20 19:35:55,434 - INFO - Completed predictions for 1332 samples
2025-11-20 19:35:55,436 - INFO - Total time: 132.02 minutes
2025-11-20 19:35:55,481 - INFO - Success: 1332, Errors: 0



✓ Completed predictions for 1332 samples
  Total time: 132.02 minutes
  Success: 1332, Errors: 0


In [14]:
# Save final predictions
results_data = {
    "predictions": predictions,
    "references": references,
    "inputs": inputs,
    "total_samples": len(test_samples),
    "timestamp": datetime.now().isoformat()
}

with open(FINAL_PREDICTIONS_FILE, "w", encoding="utf-8") as f:
    json.dump(results_data, f, indent=2, ensure_ascii=False)

logger.info(f"Final predictions saved to {FINAL_PREDICTIONS_FILE}")
print(f"Predictions saved to {FINAL_PREDICTIONS_FILE}")


2025-11-20 19:56:02,393 - INFO - Final predictions saved to deepseek_predictions.json


Predictions saved to deepseek_predictions.json


In [15]:
# Evaluation metrics (same as AWebGIS_T5_tiny notebook)

def levenshtein_similarity(preds, refs):
    """Compute Levenshtein similarity (1 - normalized edit distance)"""
    scores = []
    for pred, ref in zip(preds, refs):
        pred = pred.strip()
        ref = ref.strip()
        dist = editdistance.eval(pred, ref)
        max_len = max(len(pred), len(ref))
        scores.append(1.0 - dist / max_len if max_len > 0 else 1.0)
    return sum(scores) / len(scores)

def bleu_score(preds, refs):
    """Compute BLEU score"""
    smoothing = SmoothingFunction().method1
    vals = []
    for pred, ref in zip(preds, refs):
        pred_tokens = pred.strip().split()
        ref_tokens = ref.strip().split()
        if ref_tokens:
            vals.append(sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothing))
        else:
            vals.append(0.0)
    return sum(vals) / len(vals)

def rouge_scores(preds, refs):
    """Compute ROUGE-1 and ROUGE-L scores"""
    scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    r1, rL = [], []
    for pred, ref in zip(preds, refs):
        scores = scorer.score(ref.strip(), pred.strip())
        r1.append(scores["rouge1"].fmeasure)
        rL.append(scores["rougeL"].fmeasure)
    return {"rouge1": sum(r1) / len(r1), "rougeL": sum(rL) / len(rL)}

logger.info("Computing evaluation metrics...")
print("\nComputing evaluation metrics...")

# Compute all metrics
exact_match = load("exact_match").compute(predictions=predictions, references=references)["exact_match"]
lev_sim = levenshtein_similarity(predictions, references)
bleu = bleu_score(predictions, references)
rouge = rouge_scores(predictions, references)

# Display results
results = pd.DataFrame([{
    "Model": "DeepSeek V3.1 (via OpenRouter)",
    "Exact Match": exact_match,
    "Levenshtein Similarity": lev_sim,
    "BLEU Score": bleu,
    "ROUGE-1": rouge["rouge1"],
    "ROUGE-L": rouge["rougeL"],
}])

logger.info("Evaluation metrics computed")
logger.info(f"\n{results.to_string(index=False)}")

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
print(results.to_string(index=False))
print("\n" + "="*60)


2025-11-20 19:56:08,765 - INFO - Computing evaluation metrics...



Computing evaluation metrics...


2025-11-20 19:56:24,062 - INFO - Using default tokenizer.
2025-11-20 19:56:25,955 - INFO - Evaluation metrics computed
2025-11-20 19:56:27,144 - INFO - 
                         Model  Exact Match  Levenshtein Similarity  BLEU Score  ROUGE-1  ROUGE-L
DeepSeek V3.1 (via OpenRouter)      0.59985                0.887349    0.295903 0.886658 0.885246



EVALUATION RESULTS
                         Model  Exact Match  Levenshtein Similarity  BLEU Score  ROUGE-1  ROUGE-L
DeepSeek V3.1 (via OpenRouter)      0.59985                0.887349    0.295903 0.886658 0.885246



In [16]:
# Show some example predictions vs references
print("\nSample Predictions vs References:\n")
logger.info("Sample predictions vs references:")
for i in range(min(10, len(predictions))):
    match = '✓' if predictions[i].strip() == references[i].strip() else '✗'
    print(f"Example {i+1}:")
    print(f"  Input: {inputs[i]}")
    print(f"  Expected: {references[i]}")
    print(f"  Predicted: {predictions[i]}")
    print(f"  Match: {match}")
    print()
    logger.info(f"Example {i+1}: Match={match}, Input={inputs[i][:50]}...")


2025-11-20 20:38:08,327 - INFO - Sample predictions vs references:
2025-11-20 20:38:08,329 - INFO - Example 1: Match=✓, Input=Do a buffer on 'POLYGON((10 10, 10 20, 20 20, 20 1...
2025-11-20 20:38:08,331 - INFO - Example 2: Match=✗, Input=Run the buffer tool on the WKT 'POINT(-74.0060 40....
2025-11-20 20:38:08,332 - INFO - Example 3: Match=✓, Input=Calculate the buffer for the WKT geometry 'MULTIPO...
2025-11-20 20:38:08,335 - INFO - Example 4: Match=✗, Input=I need a 150 ft buffer zone around the line 'LINES...
2025-11-20 20:38:08,336 - INFO - Example 5: Match=✗, Input=Apply a spatial buffer of 30 feet to the geometry ...
2025-11-20 20:38:08,337 - INFO - Example 6: Match=✓, Input=Make me a buffer. Use the WKT 'POLYGON ((10 10, 20...
2025-11-20 20:38:08,341 - INFO - Example 7: Match=✓, Input=Create a buffer zone with a radius of 250 meters a...
2025-11-20 20:38:08,342 - INFO - Example 8: Match=✓, Input=Yes, please create a 2-kilometer buffer around the...
2025-11-20 20:38:08,346 - INF


Sample Predictions vs References:

Example 1:
  Input: Do a buffer on 'POLYGON((10 10, 10 20, 20 20, 20 10, 10 10))'. Distance is 5.
  Expected: BufferGeometry('POLYGON((10 10, 10 20, 20 20, 20 10, 10 10))', 5)
  Predicted: BufferGeometry('POLYGON((10 10, 10 20, 20 20, 20 10, 10 10))', 5)
  Match: ✓

Example 2:
  Input: Run the buffer tool on the WKT 'POINT(-74.0060 40.7128)' using a distance parameter of 0.25 miles.
  Expected: BufferGeometry('POINT(-74.0060 40.7128)', 402.336)
  Predicted: BufferGeometry('POINT(-74.0060 40.7128)', 0.25, 'miles')
  Match: ✗

Example 3:
  Input: Calculate the buffer for the WKT geometry 'MULTIPOINT ((10 10), (20 20))', specifying a buffer distance of 7.2.
  Expected: BufferGeometry('MULTIPOINT ((10 10), (20 20))', 7.2)
  Predicted: BufferGeometry('MULTIPOINT ((10 10), (20 20))', 7.2)
  Match: ✓

Example 4:
  Input: I need a 150 ft buffer zone around the line 'LINESTRING (30 10, 10 30, 40 40)'.
  Expected: BufferGeometry('LINESTRING (30 10, 10 30, 40 4